In [1]:
# Reinforcement Learning for Smart Traffic Signal Control
# Q-Learning Agent

import numpy as np
import random

class TrafficEnvironment:
    def __init__(self):
        self.state = None

    def reset(self):
        # State: [North-South vehicles, East-West vehicles]
        self.state = np.array([
            random.randint(0, 10),
            random.randint(0, 10)
        ])
        return self.get_state()

    def get_state(self):
        # Convert vehicle counts into simple state categories: low/high
        ns = 0 if self.state[0] < 5 else 1
        ew = 0 if self.state[1] < 5 else 1
        return (ns, ew)

    def step(self, action):
        # action 0 = North-South green signal
        # action 1 = East-West green signal

        if action == 0:
            self.state[0] = max(0, self.state[0] - random.randint(2, 5))
            self.state[1] += random.randint(0, 3)
        else:
            self.state[1] = max(0, self.state[1] - random.randint(2, 5))
            self.state[0] += random.randint(0, 3)

        # Reward: fewer waiting vehicles gives higher reward
        total_vehicles = self.state[0] + self.state[1]
        reward = -total_vehicles

        next_state = self.get_state()
        done = total_vehicles == 0

        return next_state, reward, done


# Q-learning settings
learning_rate = 0.1
discount_factor = 0.9
epsilon = 0.2
episodes = 1000

# Q-table: 4 possible states and 2 possible actions
q_table = np.zeros((2, 2, 2))

environment = TrafficEnvironment()

# Training the RL agent
for episode in range(episodes):
    state = environment.reset()

    for step in range(20):
        ns, ew = state

        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            action = random.choice([0, 1])
        else:
            action = np.argmax(q_table[ns, ew])

        next_state, reward, done = environment.step(action)
        next_ns, next_ew = next_state

        # Q-learning update rule
        old_q = q_table[ns, ew, action]
        best_next_q = np.max(q_table[next_ns, next_ew])

        q_table[ns, ew, action] = old_q + learning_rate * (
            reward + discount_factor * best_next_q - old_q
        )

        state = next_state

        if done:
            break

print("Training completed.\n")
print("Learned Q-Table:")
print(q_table)

# Test the trained agent
state = environment.reset()
print("\nInitial traffic state:", environment.state)

for step in range(10):
    ns, ew = state
    action = np.argmax(q_table[ns, ew])

    signal = "North-South Green" if action == 0 else "East-West Green"
    next_state, reward, done = environment.step(action)

    print(f"Step {step + 1}: {signal}")
    print("Vehicles waiting:", environment.state, "| Reward:", reward)

    state = next_state

    if done:
        print("Traffic cleared successfully.")
        break

Training completed.

Learned Q-Table:
[[[-39.2551285  -39.63379894]
  [-44.61225144 -40.31957927]]

 [[-51.29479946 -52.08645337]
  [-55.93458446 -55.61460251]]]

Initial traffic state: [3 6]
Step 1: East-West Green
Vehicles waiting: [5 1] | Reward: -6
Step 2: North-South Green
Vehicles waiting: [2 2] | Reward: -4
Step 3: North-South Green
Vehicles waiting: [0 3] | Reward: -3
Step 4: North-South Green
Vehicles waiting: [0 5] | Reward: -5
Step 5: East-West Green
Vehicles waiting: [0 1] | Reward: -1
Step 6: North-South Green
Vehicles waiting: [0 2] | Reward: -2
Step 7: North-South Green
Vehicles waiting: [0 5] | Reward: -5
Step 8: East-West Green
Vehicles waiting: [0 2] | Reward: -2
Step 9: North-South Green
Vehicles waiting: [0 2] | Reward: -2
Step 10: North-South Green
Vehicles waiting: [0 4] | Reward: -4
